In [ ]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

In [4]:
df_2025_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2025.01.csv", sep=';',low_memory=False) 

In [5]:
df_2025_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/Preços semestrais - AUTOMOTIVOS_2025.02.csv", sep=';',low_memory=False)

In [6]:
df_2025 = pd.concat((df_2025_01, df_2025_02))

In [8]:
df_2025.info()

<class 'pandas.DataFrame'>
Index: 813731 entries, 0 to 384207
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     804617 non-null  str    
 1   Estado - Sigla     804617 non-null  str    
 2   Municipio          804617 non-null  str    
 3   Revenda            804617 non-null  str    
 4   CNPJ da Revenda    804617 non-null  object 
 5   Nome da Rua        804617 non-null  str    
 6   Numero Rua         804535 non-null  str    
 7   Complemento        183322 non-null  str    
 8   Bairro             803093 non-null  str    
 9   Cep                804617 non-null  str    
 10  Produto            804617 non-null  str    
 11  Data da Coleta     804617 non-null  str    
 12  Valor de Venda     804617 non-null  str    
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  804617 non-null  str    
 15  Bandeira           804617 non-null  str    
dtypes: float64(1), obj

In [ ]:
df_2025_pe = df_2025[df_2025["Estado - Sigla"] == "PE"]

In [ ]:
df_2025_pe.head()

In [ ]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Staging no BigQuery
table_id = os.getenv("BRONZE_2025")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [ ]:
# Criação e carga da tabela de STG

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela de staging seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2025_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print(f"✅ Tabela Stg.{table_id} criada e dados carregados com sucesso")